In [1]:
import zipfile
import pandas as pd
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report


##################################
# Step 1: List Files in the ZIP
##################################
zip_path = 'C:/Users/Алишер/Desktop/NazarbayevUniversity/DatasetPdM/1_direct_dataset.zip'  # Update path as needed
with zipfile.ZipFile(zip_path, 'r') as zf:
    # List all files ending with .csv (case-insensitive)
    csv_files = sorted([name for name in zf.namelist() if name.lower().endswith('.csv')])

print("All CSV files in the archive:")
for f in csv_files:
    print("  ", f)

all_files = csv_files

All CSV files in the archive:
   A_B_3.5_0_0.xlsx - Sheet1.csv
   A_B_3.5_0_1.xlsx - Sheet1.csv
   A_B_3.5_1_0.xlsx - Sheet1.csv
   A_B_3.5_1_1.xlsx - Sheet1.csv
   A_B_4_0_0.xlsx - Sheet1.csv
   A_B_4_0_1.xlsx - Sheet1.csv
   A_B_4_1_0.xlsx - Sheet1.csv
   A_B_4_1_1.xlsx - Sheet1.csv
   A_B_5_0_0.xlsx - Sheet1.csv
   A_B_5_0_1.xlsx - Sheet1.csv
   A_B_5_1_0.xlsx - Sheet1.csv
   A_B_5_1_1.xlsx - Sheet1.csv
   A_B_6_0_0.xlsx - Sheet1.csv
   A_B_6_0_1.xlsx - Sheet1.csv
   A_B_6_1_0.xlsx - Sheet1.csv
   A_B_6_1_1.xlsx - Sheet1.csv


In [2]:
##################################################
# Step 2: Split Files Based on Direction Conditions
##################################################
# We assume that file names contain the substrings "A_B" and "B_A"
files_A_B = [f for f in csv_files if "A_B" in f.upper()]
# files_B_A = [f for f in csv_files if "B_A" in f.upper()]

print("\nFiles with condition A_B:")
for f in files_A_B:
    print("  ", f)
# print("\nFiles with condition B_A:")
# for f in files_B_A:
#     print("  ", f)

# For validation and test sets, we assign (for example)
# first 2 files from each condition for validation and next 2 for testing.
val_files = [files_A_B[4], files_A_B[10]]
test_files = [files_A_B[11]]

# The training set will be all files not selected for validation or testing.
assigned_files = set(val_files + test_files)
train_files = [f for f in csv_files if f not in assigned_files]

print("\nAssigned Files:")
print("Training set ({} files):".format(len(train_files)))
for f in train_files:
    print("  ", f)
print("\nValidation set ({} files):".format(len(val_files)))
for f in val_files:
    print("  ", f)
print("\nTesting set ({} files):".format(len(test_files)))
for f in test_files:
    print("  ", f)


Files with condition A_B:
   A_B_3.5_0_0.xlsx - Sheet1.csv
   A_B_3.5_0_1.xlsx - Sheet1.csv
   A_B_3.5_1_0.xlsx - Sheet1.csv
   A_B_3.5_1_1.xlsx - Sheet1.csv
   A_B_4_0_0.xlsx - Sheet1.csv
   A_B_4_0_1.xlsx - Sheet1.csv
   A_B_4_1_0.xlsx - Sheet1.csv
   A_B_4_1_1.xlsx - Sheet1.csv
   A_B_5_0_0.xlsx - Sheet1.csv
   A_B_5_0_1.xlsx - Sheet1.csv
   A_B_5_1_0.xlsx - Sheet1.csv
   A_B_5_1_1.xlsx - Sheet1.csv
   A_B_6_0_0.xlsx - Sheet1.csv
   A_B_6_0_1.xlsx - Sheet1.csv
   A_B_6_1_0.xlsx - Sheet1.csv
   A_B_6_1_1.xlsx - Sheet1.csv

Assigned Files:
Training set (13 files):
   A_B_3.5_0_0.xlsx - Sheet1.csv
   A_B_3.5_0_1.xlsx - Sheet1.csv
   A_B_3.5_1_0.xlsx - Sheet1.csv
   A_B_3.5_1_1.xlsx - Sheet1.csv
   A_B_4_0_1.xlsx - Sheet1.csv
   A_B_4_1_0.xlsx - Sheet1.csv
   A_B_4_1_1.xlsx - Sheet1.csv
   A_B_5_0_0.xlsx - Sheet1.csv
   A_B_5_0_1.xlsx - Sheet1.csv
   A_B_6_0_0.xlsx - Sheet1.csv
   A_B_6_0_1.xlsx - Sheet1.csv
   A_B_6_1_0.xlsx - Sheet1.csv
   A_B_6_1_1.xlsx - Sheet1.csv

Validation set 

In [3]:

#############################################
# Step 3: Load and Concatenate CSV Files
#############################################
def load_files_from_zip(zip_path, file_list):
    """Load CSV files specified in file_list from the zip archive and concatenate them."""
    dfs = []
    with zipfile.ZipFile(zip_path, 'r') as zf:
        for fname in file_list:
            with zf.open(fname) as f:
                df = pd.read_csv(f)
                dfs.append(df)
    return pd.concat(dfs, ignore_index=True)

df_train = load_files_from_zip(zip_path, train_files)
df_val   = load_files_from_zip(zip_path, val_files)
df_test  = load_files_from_zip(zip_path, test_files)

print("\nLoaded Data Shapes:")
print("  Training:", df_train.shape)
print("  Validation:", df_val.shape)
print("  Testing:", df_test.shape)


Loaded Data Shapes:
  Training: (379, 5)
  Validation: (32, 5)
  Testing: (13, 5)


In [ ]:
# #############################################
# # Step 4: Preprocess Data – Clean and Rename Columns
# #############################################
# # (A) Remove extra spaces from column names.
# for df in [df_train, df_val, df_test]:
#     df.columns = df.columns.str.strip()

# # (B) Rename sensor time columns so they appear as desired.
# # For example, if your CSVs originally have names like:
# # "AC_Down_to_Sort", "CB_Sort_to_Upstream", "BC_Upstream_to_Sort", "CA_Sort_to_Down"
# rename_dict = {
#     'AC_Down_to_Sort': 'AC_time',
#     'CB_Sort_to_Upstream': 'CB_time',
#     'BC_Upstream_to_Sort': 'BC_time',
#     'CA_Sort_to_Down': 'CA_time'
# }
# for df in [df_train, df_val, df_test]:
#     df.rename(columns=rename_dict, inplace=True)

# # (C) (Optional) Print column lists to verify renaming.
# print("\nTraining Set Columns:", df_train.columns.tolist())
# print("Validation Set Columns:", df_val.columns.tolist())
# print("Testing Set Columns:", df_test.columns.tolist())


In [ ]:
print(df_val)

In [4]:
#############################################
# Step 5: Create the "pressure_status" Column
#############################################
# Conditions:
#  - "low" if pressure < 4.9,
#  - "normal" if 4.9 <= pressure <= 5.2,
#  - "overpressure" if pressure > 5.2.
def pressure_status(p):
    if p < 4.9:
        return 'low'
    elif 4.9 <= p <= 5.2:
        return 'normal'
    else:
        return 'overpressure'

# IMPORTANT: Make sure the column name containing pressure values is correct.
for dataset in [df_train, df_val, df_test]:
    dataset['pressure_status'] = dataset['pressure'].apply(pressure_status)


In [5]:
#############################################
# Step 6: Define Features and Targets
#############################################
# Use the sensor time columns in the specified order:
feature_cols = ["AC_time", "CB_time"]

# Targets:
target_mech1    = 'mech_1'          # Fault indicator for sensors A->C
target_mech2    = 'mech_2'          # Fault indicator for sensors C->B
target_pressure = 'pressure_status' # Pressure category

In [6]:
#############################################
# Step 7: Train Models on the Training Set
#############################################
clf_mech1 = DecisionTreeClassifier(random_state=42)
clf_mech2 = DecisionTreeClassifier(random_state=42)
clf_pressure = DecisionTreeClassifier(random_state=42)

clf_mech1.fit(df_train[feature_cols], df_train[target_mech1])
clf_mech2.fit(df_train[feature_cols], df_train[target_mech2])
clf_pressure.fit(df_train[feature_cols], df_train[target_pressure])


DecisionTreeClassifier(random_state=42)

In [7]:
############################################
# Step 8: Evaluate Models on the Validation Set
#############################################
print("\nValidation Set Evaluation:")

print("\nClassification Report for mech_1:")
print(classification_report(df_val[target_mech1], clf_mech1.predict(df_val[feature_cols])))

print("\nClassification Report for mech_2:")
print(classification_report(df_val[target_mech2], clf_mech2.predict(df_val[feature_cols])))

print("\nClassification Report for Pressure Status:")
print(classification_report(df_val[target_pressure], clf_pressure.predict(df_val[feature_cols])))



Validation Set Evaluation:

Classification Report for mech_1:
              precision    recall  f1-score   support

           0       0.48      1.00      0.65        14
           1       1.00      0.17      0.29        18

    accuracy                           0.53        32
   macro avg       0.74      0.58      0.47        32
weighted avg       0.77      0.53      0.45        32


Classification Report for mech_2:
              precision    recall  f1-score   support

           0       1.00      0.78      0.88        32
           1       0.00      0.00      0.00         0

    accuracy                           0.78        32
   macro avg       0.50      0.39      0.44        32
weighted avg       1.00      0.78      0.88        32


Classification Report for Pressure Status:
              precision    recall  f1-score   support

         low       0.47      0.57      0.52        14
      normal       0.60      0.50      0.55        18

    accuracy                           0

c:\Anaconda\Lib\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Anaconda\Lib\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Anaconda\Lib\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [8]:
#############################################
# Step 9: Predict on the Test Set and Output Results
#############################################
pred_mech1 = clf_mech1.predict(df_test[feature_cols])
pred_mech2 = clf_mech2.predict(df_test[feature_cols])
pred_pressure = clf_pressure.predict(df_test[feature_cols])

output = {
    "mech_1_prediction": pred_mech1.tolist(),
    "mech_2_prediction": pred_mech2.tolist(),
    "pressure_prediction": pred_pressure.tolist()
}

print("\nTest Set Predictions:")
print(output)


Test Set Predictions:
{'mech_1_prediction': [0, 1, 1, 0, 1, 1, 1, 1, 1, 1, 0, 1, 0], 'mech_2_prediction': [1, 1, 0, 1, 1, 0, 1, 1, 1, 0, 1, 1, 0], 'pressure_prediction': ['low', 'low', 'low', 'low', 'low', 'low', 'low', 'low', 'low', 'low', 'normal', 'low', 'normal']}


In [9]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# Calculate accuracy for each target (convert to percentage).
acc_mech1 = accuracy_score(df_test[target_mech1], pred_mech1) * 100
acc_mech2 = accuracy_score(df_test[target_mech2], pred_mech2) * 100
acc_pressure = accuracy_score(df_test[target_pressure], pred_pressure) * 100

# Print a summary report
print("Test Set Accuracy:")
print("  Mech_1 Accuracy: {:.2f}%".format(acc_mech1))
print("  Mech_2 Accuracy: {:.2f}%".format(acc_mech2))
print("  Pressure Prediction Accuracy: {:.2f}%".format(acc_pressure))

# Optionally, you can also show full classification reports
print("\nFull Classification Report for Mech_1:")
print(classification_report(df_test[target_mech1], pred_mech1))
print("\nFull Classification Report for Mech_2:")
print(classification_report(df_test[target_mech2], pred_mech2))
print("\nFull Classification Report for Pressure Status:")
print(classification_report(df_test[target_pressure], pred_pressure))

# And even display confusion matrices if desired:
print("\nConfusion Matrix for Mech_1:")
print(confusion_matrix(df_test[target_mech1], pred_mech1))
print("\nConfusion Matrix for Mech_2:")
print(confusion_matrix(df_test[target_mech2], pred_mech2))
print("\nConfusion Matrix for Pressure Status:")
print(confusion_matrix(df_test[target_pressure], pred_pressure))


Test Set Accuracy:
  Mech_1 Accuracy: 69.23%
  Mech_2 Accuracy: 69.23%
  Pressure Prediction Accuracy: 15.38%

Full Classification Report for Mech_1:
              precision    recall  f1-score   support

           0       0.00      0.00      0.00         0
           1       1.00      0.69      0.82        13

    accuracy                           0.69        13
   macro avg       0.50      0.35      0.41        13
weighted avg       1.00      0.69      0.82        13


Full Classification Report for Mech_2:
              precision    recall  f1-score   support

           0       0.00      0.00      0.00         0
           1       1.00      0.69      0.82        13

    accuracy                           0.69        13
   macro avg       0.50      0.35      0.41        13
weighted avg       1.00      0.69      0.82        13


Full Classification Report for Pressure Status:
              precision    recall  f1-score   support

         low       0.00      0.00      0.00         

c:\Anaconda\Lib\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Anaconda\Lib\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Anaconda\Lib\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Anaconda\Lib\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning: Recall i

XGBOOST 


In [10]:
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, accuracy_score

# Initialize the label encoder and transform the pressure target values.
le = LabelEncoder()

# Encode the training target
y_train_pressure_numeric = le.fit_transform(df_train[target_pressure])
# Encode the validation and test targets using the same encoder
y_val_pressure_numeric = le.transform(df_val[target_pressure])
y_test_pressure_numeric = le.transform(df_test[target_pressure])

# Get the number of classes from the encoder.
num_classes = len(le.classes_)

# Initialize XGBoost classifiers.
clf_mech1_xgb = XGBClassifier(random_state=42, use_label_encoder=False, eval_metric='logloss', 
                              objective='binary:logistic', scale_pos_weight=1)
clf_mech2_xgb = XGBClassifier(random_state=42, use_label_encoder=False, eval_metric='logloss', 
                              objective='binary:logistic', scale_pos_weight=1)
# For the multiclass pressure model, specify the number of classes.
clf_pressure_xgb = XGBClassifier(random_state=42, use_label_encoder=False, eval_metric='mlogloss', 
                                 objective='multi:softmax', num_class=num_classes)

# Train models.
clf_mech1_xgb.fit(df_train[feature_cols], df_train[target_mech1])
clf_mech2_xgb.fit(df_train[feature_cols], df_train[target_mech2])
clf_pressure_xgb.fit(df_train[feature_cols], y_train_pressure_numeric)

# Evaluate on the validation set.
pred_mech1_val = clf_mech1_xgb.predict(df_val[feature_cols])
pred_mech2_val = clf_mech2_xgb.predict(df_val[feature_cols])
pred_pressure_val_numeric = clf_pressure_xgb.predict(df_val[feature_cols])
# Optionally convert numeric predictions back to original string labels:
pred_pressure_val = le.inverse_transform(pred_pressure_val_numeric)

print("Validation Set Accuracy:")
print("  Mech_1 Accuracy: {:.2f}%".format(accuracy_score(df_val[target_mech1], pred_mech1_val) * 100))
print("  Mech_2 Accuracy: {:.2f}%".format(accuracy_score(df_val[target_mech2], pred_mech2_val) * 100))
print("  Pressure Prediction Accuracy: {:.2f}%".format(accuracy_score(df_val[target_pressure], pred_pressure_val) * 100))

print("\nClassification Report for Pressure Status:")
print(classification_report(df_val[target_pressure], pred_pressure_val))


Validation Set Accuracy:
  Mech_1 Accuracy: 75.00%
  Mech_2 Accuracy: 78.12%
  Pressure Prediction Accuracy: 50.00%

Classification Report for Pressure Status:
              precision    recall  f1-score   support

         low       0.44      0.57      0.50        14
      normal       0.57      0.44      0.50        18

    accuracy                           0.50        32
   macro avg       0.51      0.51      0.50        32
weighted avg       0.52      0.50      0.50        32



c:\Anaconda\Lib\site-packages\xgboost\training.py:183: UserWarning: [10:58:47] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
c:\Anaconda\Lib\site-packages\xgboost\training.py:183: UserWarning: [10:58:47] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
c:\Anaconda\Lib\site-packages\xgboost\training.py:183: UserWarning: [10:58:47] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


In [11]:
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, accuracy_score

# Initialize the label encoder and transform the pressure target values.
le = LabelEncoder()

# Encode the training target
y_train_pressure_numeric = le.fit_transform(df_train[target_pressure])
# Encode the validation and test targets using the same encoder
y_val_pressure_numeric = le.transform(df_val[target_pressure])
y_test_pressure_numeric = le.transform(df_test[target_pressure])

# Get the number of classes from the encoder.
num_classes = len(le.classes_)

# Initialize XGBoost classifiers.
clf_mech1_xgb = XGBClassifier(random_state=42, use_label_encoder=False, eval_metric='logloss', 
                              objective='binary:logistic', scale_pos_weight=1)
clf_mech2_xgb = XGBClassifier(random_state=42, use_label_encoder=False, eval_metric='logloss', 
                              objective='binary:logistic', scale_pos_weight=1)
# For the multiclass pressure model, specify the number of classes.
clf_pressure_xgb = XGBClassifier(random_state=42, use_label_encoder=False, eval_metric='mlogloss', 
                                 objective='multi:softmax', num_class=num_classes)

# Train models.
clf_mech1_xgb.fit(df_train[feature_cols], df_train[target_mech1])
clf_mech2_xgb.fit(df_train[feature_cols], df_train[target_mech2])
clf_pressure_xgb.fit(df_train[feature_cols], y_train_pressure_numeric)

# Evaluate on the validation set.
pred_mech1_val = clf_mech1_xgb.predict(df_val[feature_cols])
pred_mech2_val = clf_mech2_xgb.predict(df_val[feature_cols])
pred_pressure_val_numeric = clf_pressure_xgb.predict(df_val[feature_cols])
# Optionally convert numeric predictions back to original string labels:
pred_pressure_val = le.inverse_transform(pred_pressure_val_numeric)

print("Validation Set Accuracy:")
print("  Mech_1 Accuracy: {:.2f}%".format(accuracy_score(df_val[target_mech1], pred_mech1_val) * 100))
print("  Mech_2 Accuracy: {:.2f}%".format(accuracy_score(df_val[target_mech2], pred_mech2_val) * 100))
print("  Pressure Prediction Accuracy: {:.2f}%".format(accuracy_score(df_val[target_pressure], pred_pressure_val) * 100))

print("\nClassification Report for Pressure Status:")
print(classification_report(df_val[target_pressure], pred_pressure_val))
print("Validation Set Accuracy:")
print("  Mech_1 Accuracy: {:.2f}%".format(acc_mech1))
print("  Mech_2 Accuracy: {:.2f}%".format(acc_mech2))
print("  Pressure Prediction Accuracy: {:.2f}%".format(acc_pressure))

# Print full classification reports for further details.
print("\nClassification Report for mech_1:")
print(classification_report(df_val[target_mech1], pred_mech1_val))
print("\nClassification Report for mech_2:")
print(classification_report(df_val[target_mech2], pred_mech2_val))
print("\nClassification Report for Pressure Status:")
print(classification_report(df_val[target_pressure], pred_pressure_val))



Validation Set Accuracy:
  Mech_1 Accuracy: 75.00%
  Mech_2 Accuracy: 78.12%
  Pressure Prediction Accuracy: 50.00%

Classification Report for Pressure Status:
              precision    recall  f1-score   support

         low       0.44      0.57      0.50        14
      normal       0.57      0.44      0.50        18

    accuracy                           0.50        32
   macro avg       0.51      0.51      0.50        32
weighted avg       0.52      0.50      0.50        32

Validation Set Accuracy:
  Mech_1 Accuracy: 69.23%
  Mech_2 Accuracy: 69.23%
  Pressure Prediction Accuracy: 15.38%

Classification Report for mech_1:
              precision    recall  f1-score   support

           0       0.64      1.00      0.78        14
           1       1.00      0.56      0.71        18

    accuracy                           0.75        32
   macro avg       0.82      0.78      0.75        32
weighted avg       0.84      0.75      0.74        32


Classification Report for mech_2:


c:\Anaconda\Lib\site-packages\xgboost\training.py:183: UserWarning: [10:58:53] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
c:\Anaconda\Lib\site-packages\xgboost\training.py:183: UserWarning: [10:58:53] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
c:\Anaconda\Lib\site-packages\xgboost\training.py:183: UserWarning: [10:58:53] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
c:\Anaconda\Lib\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.c

In [ ]:
# --- Step 4: Predict on the Test Set and Report Results ---
pred_mech1_test = clf_mech1_xgb.predict(df_test[feature_cols])
pred_mech2_test = clf_mech2_xgb.predict(df_test[feature_cols])


pred_pressure_test_numeric = clf_pressure_xgb.predict(df_test[feature_cols])
pred_pressure_test = le.inverse_transform(pred_pressure_test_numeric)

# Accuracy
acc_mech1_test = accuracy_score(df_test[target_mech1], pred_mech1_test) * 100
acc_mech2_test = accuracy_score(df_test[target_mech2], pred_mech2_test) * 100
acc_pressure_test = accuracy_score(df_test[target_pressure], pred_pressure_test) * 100

print("\n Test Set Accuracy:")
print("  Mech_1 Accuracy: {:.2f}%".format(acc_mech1_test))
print("  Mech_2 Accuracy: {:.2f}%".format(acc_mech2_test))
print("  Pressure Prediction Accuracy: {:.2f}%".format(acc_pressure_test))

# Optional: output predictions
output = {
    "mech_1_prediction": pred_mech1_test.tolist(),
    "mech_2_prediction": pred_mech2_test.tolist(),
    "pressure_prediction": pred_pressure_test.tolist()
}

print("\n Test Set Predictions:")
print(output)
